# 01 — Features & Target

Turns the raw caches from `00_acquire_data` into a **prediction-time-safe** feature
matrix and the continuous forward-return targets.

**Lagging** (all in one place, in `src/features.py`):
- market-derived — rolling returns, rolling volatility, technical indicators, and the
  daily FRED series — are lagged **1 trading day** (they include day-T close info).
- calendar / scheduled — day-of-week, holiday flags, `days_since_fomc` — are **not**
  lagged (known before the open).
- lower-frequency FRED (weekly/monthly/quarterly) is **not** lagged (documented
  approximation).

**Target.** Stored as the continuous forward return `fwd_ret_{H}`. Classification
buckets are **fit per fold on training data only** in stage 02 — never on the whole
sample — which fixes the look-ahead in the original quantile bucketing.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import load_config
from src.data import load_or_fetch, apply_vvix_gate
from src.features import build_feature_matrix
from src.targets import build_targets, fit_bucket_edges, apply_buckets

from index_ticker import SP500
from stockstats_technicals import STOCKSTATS_TECHNICALS

cfg = load_config(PROJECT_ROOT / "config.yaml")

def _require(path, name):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"{name} not found at {path}.\nRun 00_acquire_data.ipynb first."
        )
    return path

panel = pd.read_parquet(_require(cfg.indices_path, "indices_raw"))
panel = apply_vvix_gate(panel, cfg.start, vvix_min_start=cfg.vvix_min_start)
macro = pd.read_parquet(_require(cfg.macro_path, "macro_raw"))
fomc = (pd.read_csv(_require(cfg.fomc_path, "fomc_calendar"), parse_dates=["date"])
        if cfg.use_fomc else None)

print("panel:", panel.shape, "| macro:", macro.shape,
      "| fomc:", None if fomc is None else fomc.shape)

[vvix] start 2000-01-01 < 2007-01-01: dropped 5 VVIX column(s)
panel: (6661, 90) | macro: (9690, 24) | fomc: (370, 3)


## 1 — Feature matrix

Cached to `cache/features.parquet`. **Changing any `features:` setting in
`config.yaml` requires `force_refresh: true`** to rebuild (the cache key is just the
filename).

In [2]:
X = load_or_fetch(
    cfg.features_path,
    lambda: build_feature_matrix(
        panel, macro, fomc,
        sp_ticker=SP500,
        start=cfg.start, end=cfg.end,
        return_windows=cfg.return_windows,
        vol_windows=cfg.vol_windows,
        technicals=STOCKSTATS_TECHNICALS,
        use_technicals=cfg.use_technicals,
        use_fomc=cfg.use_fomc,
    ),
    force_refresh=cfg.force_refresh,
)

X.iloc[:3, :6]

[cache] MISS  features.parquet  (not cached) -> fetching...


c:\Users\jackshang\Desktop\Projects\sp500-prediction\src\feature_utils.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[lagged_col_name] = df[col].shift(lag_period)
c:\Users\jackshang\Desktop\Projects\sp500-prediction\src\feature_utils.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[lagged_col_name] = df[col].shift(lag_period)
c:\Users\jackshang\Desktop\Projects\sp500-prediction\src\feature_utils.py:121: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.ins

[technicals] computed 51/51 indicators for ^GSPC
[features] matrix: 6661 rows x 266 cols (2000-01-03 -> 2026-06-29)
[cache] SAVE  features.parquet  (6,661 rows x 266 cols)


,Ret_1_lag1_000001.SS,Ret_1_lag1_DX-Y.NYB,Ret_1_lag1_^BVSP,Ret_1_lag1_^DJI,Ret_1_lag1_^FCHI,Ret_1_lag1_^FTSE
date,,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,0.029117,0.001896,-0.063733,-0.03166,-0.041463,-0.038137


## 2 — Targets (continuous forward returns)

One column per horizon (`fwd_ret_5`, `fwd_ret_21`, `fwd_ret_63`). The last H rows of
each are NaN (no future) — kept so the matrix can still be used for live inference;
stage 02 drops them for training/eval.

In [3]:
close_sp = panel[("Close", SP500)]

targets = load_or_fetch(
    cfg.targets_path,
    lambda: build_targets(close_sp, cfg.horizons),
    force_refresh=cfg.force_refresh,
)

targets.describe().T

[cache] MISS  targets.parquet  (not cached) -> fetching...
[cache] SAVE  targets.parquet  (6,661 rows x 3 cols)


,count,mean,std,min,25%,50%,75%,max
fwd_ret_5,6656.0,0.001543,0.024556,-0.183401,-0.009969,0.003155,0.014622,0.191112
fwd_ret_21,6640.0,0.006395,0.047154,-0.329668,-0.016553,0.012336,0.033846,0.251144
fwd_ret_63,6598.0,0.018722,0.076709,-0.417706,-0.020213,0.030481,0.065870,0.393519


## 3 — Feature-group summary & warm-up

In [4]:
groups = {
    'returns (lagged)':    [c for c in X.columns if c.startswith('Ret_')],
    'volatility (lagged)': [c for c in X.columns if c.startswith('Vol_')],
    'technical (lagged)':  [c for c in X.columns if c.startswith('Technical_lag1_')],
    'macro':               [c for c in X.columns if c.startswith('Macro_')],
    'calendar':            [c for c in X.columns if c in ['mon','tues','wed','thurs','fri']],
    'holiday':             [c for c in X.columns if c.endswith('_holiday')],
    'fomc':                [c for c in X.columns if c == 'days_since_fomc'],
}
for name, cols in groups.items():
    print(f'{name:22s}: {len(cols):3d}')
print(f'{"TOTAL":22s}: {X.shape[1]:3d} features x {X.shape[0]} rows')

first_valid = X.dropna().index.min()
print(f'\nfirst fully-populated row: {first_valid.date()} '
      f'({int((X.index < first_valid).sum())} warm-up rows contain some NaN)')

vix_cols = [c for c in X.columns if 'VVIX' in c]
print(f'VVIX-derived features present: {len(vix_cols)} '
      f'(start {cfg.start} vs vvix_min_start {cfg.vvix_min_start})')

returns (lagged)      : 108
volatility (lagged)   :  72
technical (lagged)    :  51
macro                 :  26
calendar              :   5
holiday               :   3
fomc                  :   1
TOTAL                 : 266 features x 6661 rows

first fully-populated row: 2007-09-21 (1940 warm-up rows contain some NaN)
VVIX-derived features present: 0 (start 2000-01-01 vs vvix_min_start 2007-01-01)


## 4 — Leakage-free bucketing (demonstration)

The original derived quantile buckets from the **whole** sample, leaking future
information into the class boundaries. The fix: fit edges on the **training slice
only**, then apply them to test. Edges are open-ended (`-inf..+inf`) so out-of-sample
extremes still bucket instead of becoming NaN. This is a diagnostic here; stage 02
does it per walk-forward fold.

In [5]:
H = cfg.horizons['month']            # 21 trading days
y_cont = targets[f'fwd_ret_{H}']

dataset = pd.concat([X, y_cont.rename('y')], axis=1).dropna()
split = int(len(dataset) * 0.7)
train, test = dataset.iloc[:split], dataset.iloc[split:]

edges = fit_bucket_edges(
    train['y'], n_buckets=cfg.n_buckets,
    method=cfg.bucket_method, fixed_bins=cfg.fixed_bins,
)
train_lab = apply_buckets(train['y'], edges)
test_lab  = apply_buckets(test['y'], edges)   # SAME train-fit edges

print('bucket edges (train-fit):', [f'{e:+.4f}' for e in edges])
print('\ntrain class balance:')
print(train_lab.value_counts().sort_index().to_string())
print('\ntest class balance (train-fit edges -> naturally uneven, as expected):')
print(test_lab.value_counts().sort_index().to_string())

bucket edges (train-fit): ['-inf', '-0.0233', '+0.0051', '+0.0218', '+0.0383', '+inf']

train class balance:
y
0    658
1    658
2    658
3    658
4    658

test class balance (train-fit edges -> naturally uneven, as expected):
y
0    267
1    254
2    248
3    264
4    377
